# 01 — Azimuth partition into fold blocks

This notebook confirms that `FoldPlanner` partitions the azimuth extent into `n_folds` contiguous, non-overlapping, exhaustive blocks. The block boundaries come from `FoldPlanner._partition`, which is the foundation every fold assignment is built on.

Modules exercised: `configuration.cross_validation_config.CrossValidationConfig`, `pipelines.cross_validation_pipeline.folds.FoldPlanner`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except Exception:
    torch = None

SEED = 0
RNG  = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.titlesize" : 11,
    "axes.labelsize" : 10,
    "image.cmap"     : "viridis",
})

print("Bootstrap complete. Repository root on sys.path:", Path("../..").resolve())


## Construct a planner over a synthetic azimuth extent

The real configuration spans azimuth `[1000, 16000)`. We keep the same field names but shrink the extent so the blocks are easy to read by eye. The range extent is synthetic and only used to satisfy `CropRegion`.

In [ ]:
from configuration.cross_validation_config import CrossValidationConfig
from pipelines.cross_validation_pipeline.folds import FoldPlanner

config                     = CrossValidationConfig()
config.folds.n_folds       = 8
config.folds.azimuth_start = 0
config.folds.azimuth_end   = 800

RANGE_START = 0
RANGE_END   = 100

planner = FoldPlanner(config, range_start=RANGE_START, range_end=RANGE_END)
blocks  = planner.blocks

for index, (start, end) in enumerate(blocks):
    print(f"block {index}: [{start:4d}, {end:4d})  width {end - start}")

## Verify the partition properties

A correct partition is contiguous (block `i` ends exactly where block `i + 1` begins), covers the full extent, and never overlaps.

In [ ]:
starts = [start for start, _ in blocks]
ends   = [end   for _, end   in blocks]

contiguous = all(ends[i] == starts[i + 1] for i in range(len(blocks) - 1))
exhaustive = (starts[0] == config.folds.azimuth_start) and (ends[-1] == config.folds.azimuth_end)

print("contiguous boundaries :", contiguous)
print("covers full extent    :", exhaustive)
print("number of blocks      :", len(blocks), "(expected", config.folds.n_folds, ")")

## Visualise the blocks along the azimuth axis

Each block is drawn as a coloured band. Adjacent bands touch without gaps or overlaps, which is the visual signature of a valid partition.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 1.8))
colors  = plt.cm.tab10(np.linspace(0, 1, len(blocks)))

for index, (start, end) in enumerate(blocks):
    ax.axvspan(start, end, color=colors[index], alpha=0.7)
    ax.text((start + end) / 2, 0.5, str(index), ha="center", va="center")

ax.set_xlim(config.folds.azimuth_start, config.folds.azimuth_end)
ax.set_yticks([])
ax.set_xlabel("azimuth (samples)")
ax.set_title("Azimuth partition into fold blocks")
plt.show()

## Expected visual outcome

Eight coloured bands tile the azimuth axis end to end with no gaps and no overlaps. The printed checks report `contiguous = True` and `covers full extent = True`, and the block count equals `n_folds`.